# 🫁 Optimized Chest X-ray Classification Pipeline (Focal Loss & Patient-wise Split)

This notebook implements a state-of-the-art medical imaging pipeline with the following key features:
1.  **Patient-wise Split**: Prevents data leakage by ensuring images from the same patient aren't shared across train/val/test.
2.  **Hybrid Architecture**: Combines DenseNet121 and Swin Transformer with Spatial Attention.
3.  **Focal Loss**: Specifically handles extreme class imbalance in NIH dataset.
4.  **Interactive Epoch Training**: Allows for human-in-the-loop validation between epochs.

---

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, f1_score
from PIL import Image
from tqdm.auto import tqdm
import torch.nn.functional as F
import sys

# --- CONFIGURATION ---
DATA_DIR = 'data/images' 
METADATA_PATH = 'data/metadata_filtered.csv'
RAW_METADATA_PATH = 'data/Data_Entry_2017_v2020.csv'
TARGET_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 
    'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
IMG_SIZE = 384
BATCH_SIZE = 8
EPOCHS = 10
NUM_WORKERS = 2
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))

print(f"🚀 Device: {device} | Paths Fixed | Workers: {NUM_WORKERS}")

## 1. Loss Function (Focal Loss)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss
        if self.reduction == 'mean': return torch.mean(F_loss)
        elif self.reduction == 'sum': return torch.sum(F_loss)
        else: return F_loss

## 2. Model Architecture

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        res = torch.cat([avg_out, max_out], dim=1)
        res = self.conv(res)
        return x * self.sigmoid(res)

class HybridOptimizedModel(nn.Module):
    def __init__(self, num_classes=len(TARGET_CLASSES)):
        super().__init__()
        cnn = models.densenet121(weights='IMAGENET1K_V1')
        self.cnn_features = cnn.features
        self.cnn_pool = nn.AdaptiveAvgPool2d(1)
        self.vit = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0, img_size=IMG_SIZE)
        self.spatial_att = SpatialAttention()
        self.classifier = nn.Sequential(
            nn.Linear(1024 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.final_fc = nn.Linear(512, num_classes)

    def forward(self, x):
        c_feat = self.cnn_features(x)
        c_feat = self.spatial_att(c_feat)
        c_feat = self.cnn_pool(c_feat).view(x.size(0), -1) 
        v_feat = self.vit(x)
        combined = torch.cat([c_feat, v_feat], dim=1)
        out = self.classifier(combined)
        return self.final_fc(out)

## 3. Dataset and Patient-wise Split

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        try:
            img_name = self.dataframe.iloc[idx]['Image Index']
            img_path = os.path.join(self.root_dir, img_name)
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None: return torch.zeros((3, IMG_SIZE, IMG_SIZE)), torch.zeros(len(TARGET_CLASSES))
            
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
            # CLAHE inside __getitem__ to avoid pickling error on macOS
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            image = clahe.apply(image)
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            image = Image.fromarray(image)
            
            label = torch.tensor(self.dataframe.iloc[idx]['label_vec'], dtype=torch.float32)
            if self.transform: image = self.transform(image)
            return image, label
        except: return torch.zeros((3, IMG_SIZE, IMG_SIZE)), torch.zeros(len(TARGET_CLASSES))

def get_dataloaders():
    df = pd.read_csv(METADATA_PATH)
    
    # Get Patient ID
    if 'Patient ID' not in df.columns:
        raw_df = pd.read_csv(RAW_METADATA_PATH, usecols=['Image Index', 'Patient ID'])
        df = df.merge(raw_df, on='Image Index', how='left')

    # Filter missing
    available = set(os.listdir(DATA_DIR))
    df = df[df['Image Index'].isin(available)].reset_index(drop=True)
    
    def encode(l): return [1 if c in str(l).split('|') else 0 for c in TARGET_CLASSES]
    df['label_vec'] = df['Finding Labels'].apply(encode)
    
    unique_patients = df['Patient ID'].unique()
    np.random.seed(42)
    np.random.shuffle(unique_patients)
    
    train_size = int(0.8 * len(unique_patients))
    val_size = int(0.1 * len(unique_patients))
    
    train_patients = unique_patients[:train_size]
    val_patients = unique_patients[train_size:train_size+val_size]
    
    train_df = df[df['Patient ID'].isin(train_patients)].reset_index(drop=True)
    val_df = df[df['Patient ID'].isin(val_patients)].reset_index(drop=True)
    
    # Transformation
    train_transform = transforms.Compose([
        transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(ChestXrayDataset(train_df, DATA_DIR, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(ChestXrayDataset(val_df, DATA_DIR, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    return train_loader, val_loader

## 4. Interactive Training Engine

In [ ]:
train_loader, val_loader = get_dataloaders()
model = HybridOptimizedModel().to(device)
criterion = FocalLoss().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc = 0

def run_epoch(epoch_num):
    global best_auc
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch_num}/{EPOCHS}")
    
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
    
    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Validating"):
            out = torch.sigmoid(model(imgs.to(device)))
            all_preds.append(out.cpu().numpy())
            all_labels.append(labels.numpy())
    
    auc = 0
    try:
        y_true = np.vstack(all_labels)
        y_pred = np.vstack(all_preds)
        auc = roc_auc_score(y_true, y_pred, average='macro', multi_class='ovr')
    except Exception as e:
        print(f"\n⚠️ AUC Calculation failed: {e}")

    print(f"\n✅ Epoch {epoch_num} Results: Macro AUC: {auc:.4f} | Avg Loss: {total_loss/len(train_loader):.4f}")
    
    if auc > best_auc and auc > 0:
        best_auc = auc
        torch.save(model.state_dict(), "optimized_focal_best_notebook.pth")
        print("⭐ Best Model Saved")
    
    scheduler.step()

## 5. Main Loop (Interactive)
Run this cell to train one epoch at a time. It will ask for confirmation before proceeding.

In [ ]:
for e in range(1, EPOCHS + 1):
    run_epoch(e)
    
    if e < EPOCHS:
        val = input(f"\nEpoch {e} finished. Type 'ok' to continue to Epoch {e+1}: ")
        if val.lower() != 'ok':
            print("Training paused/stopped.")
            break